# YOLOv1 Detection Training (Pascal VOC 2012)

End-to-end YOLOv1 fine-tune following the Redmon et al. 2016 recipe, starting from
the backbone checkpoint `yolo-backbone-only.npz` produced by
`YOLO_backbone_insertion.ipynb`.

- Optimizer: SGD + momentum 0.9 + weight decay 5e-4
- LR schedule compressed to 60 epochs (1 warmup + 34 @ 1e-2 + 15 @ 1e-3 + 10 @ 1e-4)
- Paper augmentation (scale/translate +/- 20%, HSV jitter x1.5) on GPU
- Per-epoch checkpoints pushed to Google Drive


In [ ]:
!rm -rf /content/yolov1-cupy
!git clone https://github.com/mihnea-popescu/yolov1-cupy.git

import sys
sys.path.append('/content/yolov1-cupy')

from main import TestClass
test = TestClass()
test.test()


In [ ]:
# Pascal VOC 2012 dataset download + extract (Kaggle)
!curl -L -o /content/pascal-voc-2012-dataset.zip https://www.kaggle.com/api/v1/datasets/download/gopalbhattrai/pascal-voc-2012-dataset
print("Extracting Pascal VOC dataset (quiet unzip)...")
!unzip -q /content/pascal-voc-2012-dataset.zip -d /content/yolov1-cupy
!rm /content/pascal-voc-2012-dataset.zip
print("Pascal VOC dataset ready under /content/yolov1-cupy (VOC2012_train_val / VOC2012_test)")


In [ ]:
# Mount Google Drive and copy the backbone-only checkpoint locally.
# Copying to local avoids pinning npz reads through Drive's FUSE layer.
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

DRIVE_CKPT_PATH = "/content/drive/MyDrive/yolov1-cupy/yolo-backbone-only.npz"
LOCAL_CKPT_PATH = "/content/yolov1-cupy/yolo-backbone-only.npz"

assert os.path.isfile(DRIVE_CKPT_PATH), f"Not found in Drive: {DRIVE_CKPT_PATH}"
shutil.copy2(DRIVE_CKPT_PATH, LOCAL_CKPT_PATH)
print("Backbone checkpoint copied to", LOCAL_CKPT_PATH)

DRIVE_CKPT_DIR = "/content/drive/MyDrive/yolov1-cupy/checkpoints"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

LOCAL_CKPT_DIR = "/content/yolov1-cupy/models"
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)

LOCAL_LOG_PATH = "/content/yolov1-cupy/models/yolo_training_logs.log"
DRIVE_LOG_PATH = "/content/drive/MyDrive/yolov1-cupy/checkpoints/yolo_training_logs.log"


In [ ]:
# Build YOLOv1 model, load backbone weights, allocate optimizer buffers.
from yolo import YOLO

yolo = YOLO(num_classes=20)
yolo.load_weights(LOCAL_CKPT_PATH)
yolo.init_optimizer()
print("YOLO model ready; optimizer initialized.")


In [ ]:
# Hyperparameters (paper, with 60-epoch compressed schedule).
BATCH_SIZE   = 64
NUM_EPOCHS   = 60
MOMENTUM     = 0.9
WEIGHT_DECAY = 5e-4
LAMBDA_COORD = 5.0
LAMBDA_NOOBJ = 0.5

REPO = "/content/yolov1-cupy"
VOC_DATA_ROOT = "/content/yolov1-cupy"

S, B, C = 7, 2, 20
IMG_SIZE = (448, 448)


In [ ]:
# LR schedule: compressed paper ratio, 60 epochs total.
# Epoch 0: warmup 1e-3 -> 1e-2 linearly over one epoch
# Epochs 1..34:  1e-2  (34 epochs)
# Epochs 35..49: 1e-3  (15 epochs)
# Epochs 50..59: 1e-4  (10 epochs)
def lr_at(epoch, step, steps_per_epoch):
    if epoch == 0:
        return 1e-3 + (1e-2 - 1e-3) * (step / max(steps_per_epoch, 1))
    if epoch < 35:
        return 1e-2
    if epoch < 50:
        return 1e-3
    return 1e-4


In [ ]:
# Dataset sizing and batch counts.
from image_batch_loader import voc_num_batches_per_epoch

n_train_batches = voc_num_batches_per_epoch(
    REPO, BATCH_SIZE, data_root=VOC_DATA_ROOT, split="train",
)
n_val_batches = voc_num_batches_per_epoch(
    REPO, BATCH_SIZE, data_root=VOC_DATA_ROOT, split="val",
)
print(f"train batches/epoch: {n_train_batches}")
print(f"  val batches/epoch: {n_val_batches}")


In [ ]:
# Training + validation loop. Uses BatchPrefetcher so CPU data prep
# overlaps the GPU forward/backward/update of the current batch.
import time
import cupy as cp
from tqdm import tqdm

from loss import yolo_loss, yolo_loss_grad
from image_batch_loader import BatchPrefetcher

def _log_line(line):
    print(line)
    with open(LOCAL_LOG_PATH, "a") as handle:
        handle.write(line + "\n")
    try:
        shutil.copy2(LOCAL_LOG_PATH, DRIVE_LOG_PATH)
    except Exception:
        pass

for epoch in range(NUM_EPOCHS):
    epoch_started = time.time()

    yolo.train()
    train_loss_sum = 0.0
    train_loss_count = 0

    train_pref = BatchPrefetcher(
        REPO, BATCH_SIZE,
        seed=epoch, data_root=VOC_DATA_ROOT, split="train",
        n_batches=n_train_batches,
        size=IMG_SIZE, s=S, b=B, c=C,
        augment=True,
    )
    try:
        for step, (x, y) in enumerate(tqdm(train_pref, total=n_train_batches, desc=f"epoch {epoch+1}/{NUM_EPOCHS}")):
            lr = lr_at(epoch, step, n_train_batches)
            yolo.zero_grad()
            logits = yolo.forward(x)
            loss = yolo_loss(
                logits, y,
                S=S, B=B, C=C,
                lambda_coord=LAMBDA_COORD, lambda_noobj=LAMBDA_NOOBJ,
            )
            grad = yolo_loss_grad(
                logits, y,
                S=S, B=B, C=C,
                lambda_coord=LAMBDA_COORD, lambda_noobj=LAMBDA_NOOBJ,
            )
            yolo.backward(grad)
            yolo.sgd_momentum_step(lr, MOMENTUM, WEIGHT_DECAY)
            train_loss_sum += float(loss)
            train_loss_count += 1
    finally:
        train_pref.close()

    yolo.eval()
    val_loss_sum = 0.0
    val_loss_count = 0
    val_pref = BatchPrefetcher(
        REPO, BATCH_SIZE,
        seed=0, data_root=VOC_DATA_ROOT, split="val",
        n_batches=n_val_batches,
        size=IMG_SIZE, s=S, b=B, c=C,
        augment=False,
    )
    try:
        for x, y in val_pref:
            logits = yolo.forward(x)
            val_loss_sum += float(yolo_loss(
                logits, y,
                S=S, B=B, C=C,
                lambda_coord=LAMBDA_COORD, lambda_noobj=LAMBDA_NOOBJ,
            ))
            val_loss_count += 1
    finally:
        val_pref.close()

    epoch_elapsed = time.time() - epoch_started
    train_avg = train_loss_sum / max(train_loss_count, 1)
    val_avg = val_loss_sum / max(val_loss_count, 1)
    _log_line(
        f"epoch {epoch+1:03d}/{NUM_EPOCHS} "
        f"train_loss={train_avg:.4f} val_loss={val_avg:.4f} "
        f"lr_end={lr_at(epoch, n_train_batches - 1, n_train_batches):.1e} "
        f"time={epoch_elapsed:.1f}s"
    )

    local_ckpt = os.path.join(LOCAL_CKPT_DIR, f"yolo-epoch{epoch+1}.npz")
    drive_ckpt = os.path.join(DRIVE_CKPT_DIR, f"yolo-epoch{epoch+1}.npz")
    yolo.save_weights(local_ckpt)
    shutil.copy2(local_ckpt, drive_ckpt)
    print(f"  saved: {local_ckpt} -> {drive_ckpt}")
